# PROPHET MODEL TRAINING

**Notebook 04**: Train Prophet models for carbon emission forecasting

Prophet is excellent for:
- Handling trend and seasonality
- Robust to missing data
- Fast training

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import joblib
import json
from tqdm import tqdm

warnings.filterwarnings('ignore')
print("Libraries imported successfully")

/Users/lex/Desktop/Epics ki ma ka /.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully


## Load Preprocessed Data

In [2]:
train = pd.read_csv('../data/processed/train_data.csv')
test = pd.read_csv('../data/processed/test_data.csv')

print(f"Train: {len(train)} records")
print(f"Test: {len(test)} records")

Train: 53391 records
Test: 6510 records


## Select Key Series

In [3]:
# Top 10 states
top_states = test[test['fuel-name'] == 'All Fuels'].groupby('state-name')['value'].sum().nlargest(10).index.tolist()

train_subset = train[
    (train['state-name'].isin(top_states)) &
    (train['fuel-name'] == 'All Fuels')
].copy()

test_subset = test[
    (test['state-name'].isin(top_states)) &
    (test['fuel-name'] == 'All Fuels')
].copy()

print(f"Selected {len(top_states)} states")

Selected 10 states


## Prophet Model Training

In [4]:
def train_prophet_model(train_df, test_df):
    """
    Train Prophet model
    Prophet requires columns: ds (date), y (value)
    """
    try:
        # Prepare data for Prophet
        prophet_train = pd.DataFrame({
            'ds': pd.to_datetime(train_df['year'], format='%Y'),
            'y': train_df['value']
        })
        
        # Initialize and fit model
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            changepoint_prior_scale=0.05
        )
        model.fit(prophet_train)
        
        # Make future dataframe
        future = pd.DataFrame({
            'ds': pd.to_datetime(test_df['year'], format='%Y')
        })
        
        # Predict
        forecast = model.predict(future)
        predictions = forecast['yhat'].values
        
        # Calculate metrics
        mae = mean_absolute_error(test_df['value'], predictions)
        rmse = np.sqrt(mean_squared_error(test_df['value'], predictions))
        mape = mean_absolute_percentage_error(test_df['value'], predictions) * 100
        
        return {
            'model': model,
            'predictions': predictions,
            'mae': mae,
            'rmse': rmse,
            'mape': mape,
            'success': True
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

print("Prophet training function defined")

Prophet training function defined


In [5]:
# Train models
results = []
models_dict = {}

unique_combos = train_subset.groupby(['state-name', 'sector-name'])

print(f"Training Prophet models for {len(unique_combos)} series...")
print("This will take 10-15 minutes...\n")

for (state, sector), group in tqdm(unique_combos, desc="Training Prophet"):
    # Get test data
    test_group = test_subset[
        (test_subset['state-name'] == state) &
        (test_subset['sector-name'] == sector)
    ].sort_values('year')
    
    if len(test_group) == 0:
        continue
    
    train_data = group.sort_values('year')
    
    # Train model
    result = train_prophet_model(train_data, test_group)
    
    if result['success']:
        model_key = f"{state}_{sector}_All Fuels".replace(' ', '_').replace('/', '_')
        models_dict[model_key] = result['model']
        
        results.append({
            'state': state,
            'sector': sector,
            'fuel': 'All Fuels',
            'mae': result['mae'],
            'rmse': result['rmse'],
            'mape': result['mape']
        })

results_df = pd.DataFrame(results)
print(f"\nSuccessfully trained {len(results_df)} Prophet models")

Training Prophet models for 60 series...
This will take 10-15 minutes...



Training Prophet:   0%|          | 0/60 [00:00<?, ?it/s]20:47:28 - cmdstanpy - INFO - Chain [1] start processing
20:47:29 - cmdstanpy - INFO - Chain [1] done processing
Training Prophet:   2%|▏         | 1/60 [00:01<01:19,  1.34s/it]20:47:29 - cmdstanpy - INFO - Chain [1] start processing
20:47:29 - cmdstanpy - INFO - Chain [1] done processing
Training Prophet:   3%|▎         | 2/60 [00:01<00:36,  1.60it/s]20:47:29 - cmdstanpy - INFO - Chain [1] start processing
20:47:29 - cmdstanpy - INFO - Chain [1] done processing
Training Prophet:   5%|▌         | 3/60 [00:01<00:23,  2.47it/s]20:47:29 - cmdstanpy - INFO - Chain [1] start processing
20:47:29 - cmdstanpy - INFO - Chain [1] done processing
Training Prophet:   7%|▋         | 4/60 [00:01<00:17,  3.24it/s]20:47:29 - cmdstanpy - INFO - Chain [1] start processing
20:47:29 - cmdstanpy - INFO - Chain [1] done processing
20:47:29 - cmdstanpy - INFO - Chain [1] start processing
20:47:30 - cmdstanpy - INFO - Chain [1] done processing
Training P


Successfully trained 60 Prophet models


## Performance Summary

In [6]:
print("Prophet Model Performance")
print("=" * 50)
print(f"\nAverage MAPE: {results_df['mape'].mean():.2f}%")
print(f"Average MAE: {results_df['mae'].mean():.2f}")
print(f"Average RMSE: {results_df['rmse'].mean():.2f}")

print("\nTop 10 Best Performing:")
print(results_df.nsmallest(10, 'mape')[['state', 'sector', 'mape']])

Prophet Model Performance

Average MAPE: 15.07%
Average MAE: 19.08
Average RMSE: 21.64

Top 10 Best Performing:
            state                                           sector      mape
8         Florida              Industrial carbon dioxide emissions  3.120083
26      Louisiana              Industrial carbon dioxide emissions  3.264809
23        Indiana          Transportation carbon dioxide emissions  3.369606
10        Florida  Total carbon dioxide emissions from all sectors  3.439880
52          Texas  Total carbon dioxide emissions from all sectors  3.600225
58  United States  Total carbon dioxide emissions from all sectors  3.762748
56  United States              Industrial carbon dioxide emissions  4.131781
3      California             Residential carbon dioxide emissions  4.662432
59  United States          Transportation carbon dioxide emissions  4.776973
20        Indiana              Industrial carbon dioxide emissions  5.220614


## Save Models and Results

In [7]:
# Save results
results_df.to_csv('../reports/model_comparison/prophet_results.csv', index=False)
print("Saved: reports/model_comparison/prophet_results.csv")

# Save top 20 models
top_models = results_df.nsmallest(20, 'mape')

for idx, row in top_models.iterrows():
    model_key = f"{row['state']}_{row['sector']}_All Fuels".replace(' ', '_').replace('/', '_')
    if model_key in models_dict:
        filepath = f"../models/prophet/{model_key}.pkl"
        joblib.dump(models_dict[model_key], filepath)

print(f"Saved top 20 models")

# Summary
summary = {
    'model_type': 'Prophet',
    'n_models_trained': len(results_df),
    'average_mape': float(results_df['mape'].mean()),
    'average_mae': float(results_df['mae'].mean()),
    'average_rmse': float(results_df['rmse'].mean())
}

with open('../models/prophet/prophet_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)
print("Saved: models/prophet/prophet_summary.json")

Saved: reports/model_comparison/prophet_results.csv
Saved top 20 models
Saved: models/prophet/prophet_summary.json


## Generate Future Predictions

In [8]:
# Predictions for 2022-2030
future_predictions = []

for (state, sector), group in tqdm(train_subset.groupby(['state-name', 'sector-name']),
                                    desc="Generating forecasts"):
    # Full series
    full_data = pd.concat([
        train_subset[(train_subset['state-name'] == state) & (train_subset['sector-name'] == sector)],
        test_subset[(test_subset['state-name'] == state) & (test_subset['sector-name'] == sector)]
    ]).sort_values('year')
    
    try:
        prophet_data = pd.DataFrame({
            'ds': pd.to_datetime(full_data['year'], format='%Y'),
            'y': full_data['value']
        })
        
        model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        model.fit(prophet_data)
        
        future = model.make_future_dataframe(periods=9, freq='Y')
        forecast = model.predict(future)
        
        # Get predictions for 2022-2030
        future_years = forecast.tail(9)
        
        for idx, row in future_years.iterrows():
            year = row['ds'].year
            if year >= 2022 and year <= 2030:
                future_predictions.append({
                    'year': year,
                    'state': state,
                    'sector': sector,
                    'fuel': 'All Fuels',
                    'predicted_value': row['yhat'],
                    'model': 'Prophet'
                })
    except:
        pass

predictions_df = pd.DataFrame(future_predictions)
predictions_df.to_csv('../data/dashboard/prophet_predictions_2022_2030.csv', index=False)
print(f"\nSaved {len(predictions_df)} predictions")

Generating forecasts:   0%|          | 0/60 [00:00<?, ?it/s]20:48:15 - cmdstanpy - INFO - Chain [1] start processing
20:48:15 - cmdstanpy - INFO - Chain [1] done processing
Generating forecasts:   2%|▏         | 1/60 [00:00<00:09,  6.45it/s]20:48:15 - cmdstanpy - INFO - Chain [1] start processing
20:48:15 - cmdstanpy - INFO - Chain [1] done processing
Generating forecasts:   3%|▎         | 2/60 [00:00<00:08,  7.22it/s]20:48:15 - cmdstanpy - INFO - Chain [1] start processing
20:48:15 - cmdstanpy - INFO - Chain [1] done processing
Generating forecasts:   5%|▌         | 3/60 [00:00<00:07,  7.61it/s]20:48:15 - cmdstanpy - INFO - Chain [1] start processing
20:48:15 - cmdstanpy - INFO - Chain [1] done processing
Generating forecasts:   7%|▋         | 4/60 [00:00<00:07,  7.14it/s]20:48:15 - cmdstanpy - INFO - Chain [1] start processing
20:48:15 - cmdstanpy - INFO - Chain [1] done processing
Generating forecasts:   8%|▊         | 5/60 [00:00<00:07,  7.52it/s]20:48:15 - cmdstanpy - INFO - Chain


Saved 480 predictions


## Complete

In [9]:
print("=" * 70)
print("PROPHET MODEL TRAINING COMPLETE")
print("=" * 70)
print(f"\nModels trained: {len(results_df)}")
print(f"Average MAPE: {results_df['mape'].mean():.2f}%")
print("\nNext: Run 05_lstm_model.ipynb")

PROPHET MODEL TRAINING COMPLETE

Models trained: 60
Average MAPE: 15.07%

Next: Run 05_lstm_model.ipynb
